# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Astroking2004/flyrank-ml-internship1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one pseudonymized content item from the Hugging Face warehouse release. The hosted warehouse is the source of truth here, so the contract should describe the content-level grain and the date span observed in the parquet tables rather than the starter CSV shipped in the repo.


In [2]:
import os
from pathlib import Path
import duckdb


def get_hf_token():
    for key in ("HF_TOKEN", "HF_Token"):
        value = os.environ.get(key)
        if value:
            return value

    for candidate in [Path.cwd(), *Path.cwd().parents, Path("c:/Users/Public/Downloads/flyrank-ml-internship1")]:
        env_path = candidate / ".env"
        if env_path.exists():
            for line in env_path.read_text(encoding="utf-8").splitlines():
                if "=" in line:
                    key, value = line.split("=", 1)
                    key = key.strip()
                    value = value.strip().strip('"').strip("'")
                    if key in {"HF_TOKEN", "HF_Token"}:
                        return value
            break

    return None


HF_TOKEN = get_hf_token()
if not HF_TOKEN:
    print("HF token not found. Set HF_TOKEN in the environment or add it to the repo .env file.")
else:
    con = duckdb.connect()
    try:
        con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
    except Exception as exc:
        print("Secret setup failed:", exc)

    rel = 'hf://datasets/FlyRank/internship-warehouse'

    # Count the warehouse tables directly from the hosted parquet source.
    for name, src in {
        'dim_clients': f"read_parquet('{rel}/dim_clients.parquet')",
        'dim_content': f"read_parquet('{rel}/dim_content.parquet')",
        'fact_content_daily_performance': f"read_parquet('{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet')",
        'fact_content_query_90d': f"read_parquet('{rel}/fact_content_query_90d.parquet')",
    }.items():
        try:
            result = con.sql(f'SELECT COUNT(*) FROM {src}').fetchall()
            n = result[0][0] if result else 0
            print(f'{name}: {n:,} rows')
        except Exception as exc:
            print(f'{name}: query failed -> {exc}')

    print('\nGrain check on a mid-panel month partition:')
    try:
        rows = con.sql(f"""
            SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
            FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet')
            GROUP BY 1, 2, 3
            HAVING COUNT(*) > 1
            LIMIT 5
        """).fetchall()
        print(rows)
    except Exception as exc:
        print('Grain check failed:', exc)


dim_clients: 104 rows
dim_content: 519,606 rows
fact_content_daily_performance: 9,841,378 rows
fact_content_query_90d: 2,414,248 rows

Grain check on a mid-panel month partition:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[]


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features should be only the signals known before the prediction moment, such as content metadata and historical performance fields from the warehouse tables. The label is the outcome being defined for this notebook, while the hash ids stay in context for joins and grouped splits. Provider and model metadata belong in excluded because they describe provenance, not evidence.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
from pathlib import Path
import duckdb


def get_hf_token():
    for key in ("HF_TOKEN", "HF_Token"):
        value = os.environ.get(key)
        if value:
            return value

    for candidate in [Path.cwd(), *Path.cwd().parents, Path("c:/Users/Public/Downloads/flyrank-ml-internship1")]:
        env_path = candidate / ".env"
        if env_path.exists():
            for line in env_path.read_text(encoding="utf-8").splitlines():
                if "=" in line:
                    key, value = line.split("=", 1)
                    key = key.strip()
                    value = value.strip().strip('"').strip("'")
                    if key in {"HF_TOKEN", "HF_Token"}:
                        return value
            break

    return None


HF_TOKEN = get_hf_token()
if not HF_TOKEN:
    print("HF token not found. Set HF_TOKEN in the environment or add it to the repo .env file.")
else:
    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

    rel = 'hf://datasets/FlyRank/internship-warehouse'

    # Use the same warehouse tables as the contract discussion.
    query = f"""
    SELECT *
    FROM read_parquet('{rel}/dim_content.parquet')
    LIMIT 5
    """
    rows = con.sql(query).fetchall()
    print(rows)


[('client_04660893ae39614a', 'content_004de9653278b5a4', 'keyword_e754999ab88dd9f2', 'url_d6091f18cf628794', 22, 4, 108, datetime.date(2026, 5, 30), datetime.date(2026, 7, 1), 'keyword article', 30, 0.91, 'HIGH', 0.98, 'transactional', 16, 3, datetime.date(2026, 5, 12), 'gemini-generate-content', 'gemini-3-flash-preview', 15682, 2555, None, None, True, False), ('client_04660893ae39614a', 'content_00dc5efae381b2ab', 'keyword_4329d7aede8e208b', 'url_3a66d2f2e36823ca', 31, 6, 95, datetime.date(2026, 6, 12), datetime.date(2026, 7, 1), 'keyword article', 10, 0.0, 'LOW', 0.0, 'commercial', 0, 4, datetime.date(2026, 6, 1), 'gemini-generate-content', 'gemini-3-flash-preview', 15438, 2430, None, None, True, False), ('client_04660893ae39614a', 'content_01410f2556c327ac', 'keyword_9b08047d3d2a0406', 'url_809eda7a7e20b3b2', 22, 5, 82, datetime.date(2026, 5, 9), datetime.date(2026, 7, 1), 'keyword article', 480, 0.36, 'MEDIUM', 0.62, 'informational', 169, 4, datetime.date(2026, 5, 6), 'gemini-gener

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Use the query cells below to confirm the grain, counts, and missingness patterns that the markdown claims rely on. If the query output disagrees with the sentence above it, trust the query and rewrite the claim.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
from pathlib import Path
import duckdb


def get_hf_token():
    for key in ("HF_TOKEN", "HF_Token"):
        value = os.environ.get(key)
        if value:
            return value

    for candidate in [Path.cwd(), *Path.cwd().parents, Path("c:/Users/Public/Downloads/flyrank-ml-internship1")]:
        env_path = candidate / ".env"
        if env_path.exists():
            for line in env_path.read_text(encoding="utf-8").splitlines():
                if "=" in line:
                    key, value = line.split("=", 1)
                    key = key.strip()
                    value = value.strip().strip('"').strip("'")
                    if key in {"HF_TOKEN", "HF_Token"}:
                        return value
            break

    return None


HF_TOKEN = get_hf_token()
if not HF_TOKEN:
    print("HF token not found. Set HF_TOKEN in the environment or add it to the repo .env file.")
else:
    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

    rel = 'hf://datasets/FlyRank/internship-warehouse'

    print('rows in dim_content:')
    print(con.sql(f"SELECT COUNT(*) AS n FROM read_parquet('{rel}/dim_content.parquet')").fetchall())
    print('\nrows in daily performance table:')
    print(con.sql(f"SELECT COUNT(*) AS n FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet')").fetchall())
    print('\nmin/max report dates:')
    print(con.sql(f"SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet')").fetchall())
    print('\nmissingness in dim_content sample:')
    print(con.sql(f"""
    SELECT
        SUM(CASE WHEN keyword_hash_id IS NULL THEN 1 ELSE 0 END) AS missing_keyword_hash,
        SUM(CASE WHEN url_hash_id IS NULL THEN 1 ELSE 0 END) AS missing_url_hash,
        SUM(CASE WHEN content_type IS NULL THEN 1 ELSE 0 END) AS missing_content_type
    FROM read_parquet('{rel}/dim_content.parquet')
    """).fetchall())


rows in dim_content:
[(519606,)]

rows in daily performance table:
[(9841378,)]

min/max report dates:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[(datetime.date(2026, 3, 1), datetime.date(2026, 3, 31))]

missingness in dim_content sample:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[(71998, 6525, 0)]


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This warehouse release is powerful for analysis, but it still has blind spots: client history starts at different points, some early rows are only partially populated, and fixed windows can overlap the moment you want to predict. The limits section should say exactly which assumptions those gaps force.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
from pathlib import Path
import duckdb


def get_hf_token():
    for key in ("HF_TOKEN", "HF_Token"):
        value = os.environ.get(key)
        if value:
            return value

    for candidate in [Path.cwd(), *Path.cwd().parents, Path("c:/Users/Public/Downloads/flyrank-ml-internship1")]:
        env_path = candidate / ".env"
        if env_path.exists():
            for line in env_path.read_text(encoding="utf-8").splitlines():
                if "=" in line:
                    key, value = line.split("=", 1)
                    key = key.strip()
                    value = value.strip().strip('"').strip("'")
                    if key in {"HF_TOKEN", "HF_Token"}:
                        return value
            break

    return None


HF_TOKEN = get_hf_token()
if not HF_TOKEN:
    print("HF token not found. Set HF_TOKEN in the environment or add it to the repo .env file.")
else:
    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

    rel = 'hf://datasets/FlyRank/internship-warehouse'

    counts = con.sql(f"""
    SELECT COUNT(*) AS n
    FROM read_parquet('{rel}/dim_content.parquet')
    WHERE keyword_hash_id IS NULL AND url_hash_id IS NULL
    """).fetchall()
    print('rows with no keyword data in dim_content:', counts[0][0] if counts else 0)

    counts = con.sql(f"""
    SELECT COUNT(*) AS n
    FROM read_parquet('{rel}/dim_content.parquet')
    WHERE content_type IS NULL
    """).fetchall()
    print('rows with missing content_type:', counts[0][0] if counts else 0)

    print('rows with no usable history in daily facts:')
    print(con.sql(f"""
    SELECT COUNT(*) AS n
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet')
    WHERE gsc_impressions IS NULL OR ga4_pageviews IS NULL
    """).fetchall())


rows with no keyword data in dim_content: 5151
rows with missing content_type: 0
rows with no usable history in daily facts:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[(3018741,)]


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.